In [7]:
import os
import cv2 as cv
import numpy as np

In [8]:
# Using Hit & Miss method for skeletonization

def skeletonize(frame):
    
    skeleton = np.zeros(frame.shape, np.uint8)
    
    # Using a 3x3 "+" structure kernal
    kernel = cv.getStructuringElement(cv.MORPH_CROSS, (3, 3))
    
    size = np.size(frame)
    cond = True
    
    while cond:
        eroded = cv.erode(frame, kernel)
        temp = cv.dilate(eroded, kernel)
        temp = cv.subtract(frame, temp)
        
        # Image masking
        skeleton = cv.bitwise_or(skeleton, temp)
        
        frame = eroded.copy()
        zeros = size - cv.countNonZero(frame)
        if zeros == size:
            cond = False
    
    return skeleton

In [9]:
def preprocess(frame):
    
    # Apply threshold
    frame = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
    t_value, frame = cv.threshold(frame, 0, 255, cv.THRESH_BINARY_INV + cv.THRESH_OTSU)
    
    # Remove noise and enhance the subject
    kernel = np.ones((3, 3), np.uint8)
    frame = cv.morphologyEx(frame, cv.MORPH_OPEN, kernel)
    
    return t_value, frame

In [10]:
directory = "sample_movies"
index = 7
video = os.listdir(directory)[index]
cap = cv.VideoCapture(directory + "\\" + video)

# Collections of threshold values used
t_values = set()

while cap.isOpened():
    
    ret, frame = cap.read()
    if not ret:
        print("Can't receive frame (stream end?). Exiting ...")
        break
    
    # Preparation
    t, thresh = preprocess(frame)
    t_values.add(t) # add the threshold value to collection
    
    # Skeletonization
    skeleton = skeletonize(thresh)
    
    # Adjustment: Change the color of the skeleton for feature clarity
    skeleton_green = cv.cvtColor(skeleton, cv.COLOR_GRAY2BGR)
    black_threshold = 10
    white_pixels = np.all(skeleton_green > black_threshold, axis=-1)
    skeleton_green[white_pixels] = [0,255,0]
    
    # Composition of the overlay and original
    composition = frame.copy()
    composition[skeleton_green != 0] = skeleton_green[skeleton_green != 0]
    
    cv.imshow('Skeleton', composition)
    
    if cv.waitKey(1) == ord('q'):
        break

cap.release()
cv.destroyAllWindows()

Can't receive frame (stream end?). Exiting ...


In [11]:
print(f"Set of threshold values used in Video {index+1}: {t_values}")

Set of threshold values used in Video 8: {129.0, 130.0, 131.0, 132.0, 133.0, 114.0, 126.0}
